In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

# 1. Load Datasets
print("Loading data...")
train = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/train.csv")
test = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/test.csv")

# Save target and index
target = train['demand']
test_idx = test['Index']

# Combine for uniform feature engineering
df = pd.concat([train.drop(columns=['demand']), test], axis=0).reset_index(drop=True)

# 2. Feature Engineering & Handling Nulls
print("Engineering features...")

# Handle Categorical Missing Values
categorical_cols = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for col in categorical_cols:
    df[col] = df[col].fillna('MISSING').astype(str)

# Handle Numerical Missing Values
df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
df['NumberofLanes'] = df['NumberofLanes'].fillna(df['NumberofLanes'].mode()[0])

# Parse Timestamp ('15:30' -> hour=15, minute=30)
df['hour'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]))
df['minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]))

# Cyclical Encoding for Time
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60.0)
df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60.0)

# Frequency Encoding for High-Cardinality Categoricals (like geohash)
for col in ['geohash', 'RoadType']:
    freq = df[col].value_counts().to_dict()
    df[f'{col}_freq'] = df[col].map(freq)

# Convert categorical columns to category dtype for LightGBM/CatBoost native support
for col in categorical_cols:
    df[col] = df[col].astype('category')

# Separate back into Train and Test
X_train = df.iloc[:len(train)].drop(columns=['Index', 'timestamp'])
X_test = df.iloc[len(train):].drop(columns=['Index', 'timestamp'])

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# 3. Model Training & Ensembling
print("Training models...")

# Define categorical indices for CatBoost
cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_cols]

# Initialize arrays for out-of-fold and test predictions
lgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

# 5-Fold Cross-Validation Setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, target)):
    print(f"--- Training Fold {fold + 1} ---")
    
    X_tr, y_tr = X_train.iloc[train_idx], target.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], target.iloc[val_idx]
    
    # 3a. LightGBM Regressor
    lgb_model = LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=63,
        random_state=42,
        n_jobs=-1
    )
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[]  # Add early stopping if desired
    )
    lgb_test_preds += lgb_model.predict(X_test) / kf.n_splits
    
    # 3b. CatBoost Regressor
    cat_model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        cat_features=cat_features_idx,
        random_seed=42,
        verbose=0
    )
    cat_model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50
    )
    cat_test_preds += cat_model.predict(X_test) / kf.n_splits

# 4. Blend Predictions (Weighted Ensemble)
# Giving a slightly higher weight to CatBoost as it handles categorical strings beautifully
final_preds = (0.4 * lgb_test_preds) + (0.6 * cat_test_preds)

# 5. Create Submission File
submission = pd.DataFrame({
    'Index': test_idx,
    'demand': final_preds
})

# Post-processing: Ensure no negative demands (if any were predicted)
submission['demand'] = submission['demand'].clip(lower=0)

submission.to_csv('submission.csv', index=False)
print("Ensemble submission file saved successfully as 'submission.csv'!")

Loading data...
Engineering features...
Train shape: (77299, 16), Test shape: (41778, 16)
Training models...
--- Training Fold 1 ---
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001620 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1580
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 16
[LightGBM] [Info] Start training from score 0.093784
--- Training Fold 2 ---
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a lar